# DB-PatchKAN — Colab A100 training

Runs BF16 masked pretraining, Hyperband tuning, hierarchical concept-KAN fine-tuning, and a separately guarded final evaluation. Select an A100 runtime first. NPZ tensors follow the segment exchange contract and are grouped by `EVTripBagDataset` for one supervised loss row per trip. SSL still uses the fold-safe underlying segments.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/dbpatchkan
%pip install -q -e '.[tuning,explain]'

In [ ]:
import json
import subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Enable a GPU runtime first.'
print(torch.cuda.get_device_name(0), 'BF16:', torch.cuda.is_bf16_supported())
DATA = Path('/content/drive/MyDrive/dbpatchkan/segments_train.npz')
VALIDATION = Path('/content/drive/MyDrive/dbpatchkan/segments_validation.npz')
OUTPUT = Path('/content/drive/MyDrive/dbpatchkan/runs/a100_primary_seed42')
assert DATA.exists() and VALIDATION.exists(), 'Upload fold-separated train/validation tensors.'

## Optional resumable 24-trial Hyperband search
This stage receives only training and validation files. The untouched test NPZ must remain unmounted and is absent from the tuning CLI interface. Real trip bags contain 6-279 segments, so SSL uses segment batches of 256 while supervised tuning preserves every segment and uses two trips per A100 batch.

In [ ]:
RUN_TUNING = False
TUNING_OUTPUT = OUTPUT.parent / 'a100_hyperband_tuning'
BEST_CONFIG = TUNING_OUTPUT / 'best_config.json'
if RUN_TUNING:
    subprocess.run([
        'python', '-m', 'dbpatchkan.training.tune',
        '--train-data', str(DATA), '--validation-data', str(VALIDATION),
        '--trip-bags',
        '--output-dir', str(TUNING_OUTPUT),
        '--study-db', str(TUNING_OUTPUT / 'optuna_study.db'),
        '--study-name', 'dbpatchkan_a100', '--trials', '24',
        '--device', 'cuda', '--precision', 'bf16', '--domain', 'nissan',
        '--pretrain-epochs', '10', '--finetune-epochs', '20',
        '--pretrain-batch-size', '256', '--finetune-batch-size', '2',
        '--gradient-accumulation', '1', '--pretrain-lr', '3e-4',
        '--encoder-lr', '1e-5', '--head-lr', '3e-4',
        '--weight-decay', '0.01', '--gradient-clip', '1.0',
        '--num-workers', '4', '--seed', '42', '--freeze-encoders',
    ], check=True)
    print(BEST_CONFIG.read_text())

## Default-configuration reference run
This is the registered DB-PatchKAN reference configuration. Hyperparameter conclusions must use the persistent validation-only study above.

In [ ]:
command = [
    'python', '-m', 'dbpatchkan.training.train',
    '--data', str(DATA), '--validation-data', str(VALIDATION),
    '--trip-bags',
    '--output-dir', str(OUTPUT), '--stage', 'both',
    '--device', 'cuda', '--precision', 'bf16', '--domain', 'nissan',
    '--segment-length', '256', '--patch-length', '16', '--patch-stride', '8',
    '--d-model', '128', '--encoder-layers', '3', '--heads', '4', '--ff-dim', '256',
    '--dropout', '0.10', '--kan-grid', '5', '--fusion-head', 'kan',
    '--pretrain-batch-size', '256', '--finetune-batch-size', '2',
    '--pretrain-epochs', '60', '--epochs', '80', '--patience', '15',
    '--pretrain-lr', '3e-4', '--encoder-lr', '1e-5', '--head-lr', '3e-4',
    '--weight-decay', '0.01', '--warmup-fraction', '0.05',
    '--gradient-clip', '1.0', '--mask-ratio', '0.4',
    '--gradient-accumulation', '1', '--num-workers', '4',
    '--seed', '42', '--freeze-encoders',
]
subprocess.run(command, check=True)

## Five-seed final runs
Run only after configuration selection on validation folds. Keep the untouched vehicle/time test file outside every supervised and self-supervised stage.

In [ ]:
from dbpatchkan.models import (DBPatchKAN, DBPatchKANConfig, EVTripBagDataset, assert_disjoint_trip_bags)
from dbpatchkan.training.train import TrainingConfig, train
from dbpatchkan.training.tune import ConceptSubsetDataset

train_data = EVTripBagDataset.from_npz(DATA)
validation_data = EVTripBagDataset.from_npz(VALIDATION)
assert_disjoint_trip_bags(train_data, validation_data)
if BEST_CONFIG.exists():
    selected_payload = json.loads(BEST_CONFIG.read_text())
    selected_model_config = DBPatchKANConfig(**selected_payload['model_config'])
    selected_mask_ratio = selected_payload['parameters']['mask_ratio']
else:
    selected_model_config, selected_mask_ratio = DBPatchKANConfig(), 0.4
train_view = ConceptSubsetDataset(train_data, selected_model_config.n_concepts)
validation_view = ConceptSubsetDataset(validation_data, selected_model_config.n_concepts)
for seed in (42, 52, 62, 72, 82):
    final_training = TrainingConfig(
        stage='both', device='cuda', precision='bf16',
        pretrain_batch_size=256, finetune_batch_size=2, gradient_accumulation=1,
        pretrain_epochs=60, finetune_epochs=80, patience=15,
        pretrain_lr=3e-4, encoder_lr=1e-5, head_lr=3e-4,
        weight_decay=0.01, warmup_fraction=0.05, gradient_clip=1.0,
        mask_ratio=selected_mask_ratio, num_workers=4, seed=seed,
        domain='nissan', freeze_encoders=True, trip_bags=True,
    )
    train(
        DBPatchKAN(selected_model_config), train_view, validation_view, final_training,
        output_dir=OUTPUT.parent / f'a100_final_seed{seed}',
    )

## One-time untouched-test evaluation — run last
Freeze the architecture, hyperparameters, calibration domain, checkpoint-selection rule, and final seed using validation results before enabling this cell. The evaluator restores `best.pt`, fits nothing, verifies identical state fingerprints before/after inference, and refuses to overwrite an existing report. Never use these results to revise the model.

In [ ]:
RUN_FINAL_UNTOUCHED_TEST = False  # Change once only after all selection is frozen.
UNTOUCHED_TEST = Path('/content/drive/MyDrive/dbpatchkan/segments_test_UNTOUCHED.npz')
SELECTED_FINAL_SEED = 42  # Choose from validation evidence before opening test data.
SELECTED_CHECKPOINT = OUTPUT.parent / f'a100_final_seed{SELECTED_FINAL_SEED}' / 'best.pt'
TEST_REPORT = OUTPUT.parent / 'FINAL_UNTOUCHED_TEST_REPORT'
if RUN_FINAL_UNTOUCHED_TEST:
    assert SELECTED_CHECKPOINT.exists(), 'Freeze and finish the selected model first.'
    assert UNTOUCHED_TEST.exists(), 'Mount the untouched NPZ only at this final stage.'
    subprocess.run([
        'python', '-m', 'dbpatchkan.training.evaluate',
        '--checkpoint', str(SELECTED_CHECKPOINT),
        '--test-data', str(UNTOUCHED_TEST),
        '--output-dir', str(TEST_REPORT),
        '--dataset-kind', 'trip', '--domain', 'auto',
        '--device', 'cuda', '--precision', 'bf16',
        '--batch-size', '2', '--num-workers', '4',
        '--final-evaluation',
    ], check=True)

Training checkpoints are written as `pretrain_latest.pt`, `latest.pt`, and validation-selected `best.pt`. The final evaluator writes immutable-path `untouched_test_predictions.csv` and `untouched_test_metrics.json`; KAN splines remain FP32 under BF16 Transformer inference.